In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
from  dotenv import load_dotenv
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, classification_report, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import time
import pickle

warnings.filterwarnings('ignore', category=FutureWarning, message='.*idxmax.*')

In [ ]:
# pip install xgboost

# 2.1. Datos

## 2.1.1. Datos aeropuertos

In [ ]:
all_files = os.listdir('.')

csv_files = [f for f in all_files if f.startswith('T_ONTIME_REPORTING') and f.endswith('.csv')]
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index = True)

print(f"Total de filas: {len(df):,}")
print(f"Total de columnas: {df.shape[1]}")

airports = df['ORIGIN'].unique()
print(f"Total de airports únicos: {len(airports)}")
print(airports)

In [ ]:
top_airports = df['ORIGIN'].value_counts()
top_airports

### Seleccionamos los aeropuertos con más de n vuelos, sin superar los 50 aeropuertos


---



In [ ]:
for n in range (1, 1000000):
    top_50 = top_airports[top_airports > n].index.tolist()
    if 30 < len(top_50) < 50:
        break
print(f"Aeropuertos seleccionados: {len(top_50)}")
print(top_50)

De los 350 aeropuertos que hay en el dataset que hemos obtenido de todo el año 2023, se han filtrado aquellos que tienen mayor flujo, aquellos con mas de 32584 vuelos, obteniendo así un listado de 49 aeropuertos, que concentran la mayoría de tráfico doméstico en EEUU.

## 2.1.2. Datos noaa

### 2.1.2.1. Descarga de datos

In [ ]:
url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
df_stations = pd.read_csv(url, low_memory=False)


print(df_stations.head())
print(df_stations.columns.tolist())

In [ ]:
df_us = df_stations[df_stations['CTRY'] == 'US'].copy()
df_us['IATA'] = df_us['ICAO'].str[1:]
df_my_airports = df_us[df_us['IATA'].isin(top_50)].copy()
df_my_airports = df_my_airports.sort_values('END', ascending = False).drop_duplicates(subset = 'IATA', keep = 'first')
print(f"Lista de aeropuertos:")
print(df_my_airports.columns.tolist())
print(df_my_airports.head())

In [ ]:
noaa_stations = {}
for _, row in df_my_airports.iterrows():
    station_id = f"{str(row['USAF'])}{str(row['WBAN']).zfill(5)}"
    noaa_stations[row['IATA']] = station_id

print(f"Aeropuertos encontrados:")
print(noaa_stations)

In [ ]:
os.makedirs("noaa_data", exist_ok=True)

download_data = False

if download_data:
  token = os.environ.get("NOAA_TOKEN")
  headers = {"token": token}
  url = "https://www.ncei.noaa.gov/access/services/data/v1"
  for iata, station_id in noaa_stations.items():
      print(f"Descargando {iata}...")
      print(f"{iata} descargado correctamente.")
      params = {
          "dataset": "global-hourly",
          "stations": station_id,
          "startDate": "2023-01-01T00:00:00",
          "endDate": "2023-12-31T23:59:59",
          "format": "csv"
      }

      try:
          response = requests.get(url, headers=headers, params=params, timeout=60)

          if response.status_code == 200:
              filepath = f"noaa_data/{iata}_2023.csv"
              with open(filepath, "wb") as f:
                  f.write(response.content)
              print(f"{iata} descargado correctamente.")
          else:
              print(f"{iata} error: {response.status_code}.")

      except Exception as e:
          print(f"{iata} excepción: {e}.")

      # Pausa para no saturar la API
      time.sleep(2)
  print("Descarga completada.")
else:
  print("Los datos ya han sido descargados previamente.")

# 2.2 Limpieza NOAA

In [ ]:
NOAA_PATH = "../data/noaa_data"

def parse_escalated_value(field, scale=10, missing_value="+9999"):
    """
    Extrae el primer valor numérico de un campo ISD tipo 'TMP', 'DEW', 'SLP',
    que viene como 'signo + digitos, codigo_calidad'. Divide por la escala indicada.
    Devuelve NaN si el valor es el código de 'missing' de NOAA (+9999, 999,9, etc.)
    """
    if pd.isna(field):
        return np.nan
    value_str = str(field).split(',')[0]
    if missing_value in value_str or '9999' in value_str:
        return np.nan
    try:
        return int(value_str) / scale
    except ValueError:
        return np.nan

def parse_wind(field):
    """
    Campo WND: 'direccion,codigo_calidad_dir,tipo,velocidad,codigo_calidad_vel'
    Velocidad viene en m/s * 10. Devuelve velocidad en m/s.
    """
    if pd.isna(field):
        return np.nan
    parts = str(field).split(',')
    if len(parts) < 4:
        return np.nan
    speed_str = parts[3]
    if speed_str == '9999':
        return np.nan
    try:
        return int(speed_str) / 10
    except ValueError:
        return np.nan

def parse_precipitation(field):
    """
    Campo AA1: 'periodo_horas,cantidad_mm*10,condicion,codigo_calidad'
    Devuelve la cantidad de precipitación en mm. Si no hay AA1, asumimos 0 (no llovió).
    """
    if pd.isna(field):
        return 0.0
    parts = str(field).split(',')
    if len(parts) < 2:
        return 0.0
    amount_str = parts[1]
    if amount_str == '9999':
        return np.nan
    try:
        return int(amount_str) / 10
    except ValueError:
        return np.nan


def clean_noaa(iata, folder_path=NOAA_PATH):
    """
    Carga y limpia el csv NOAA de un aeropuerto, agregando los datos a nivel horario
    (nos quedamos con una fila por hora, la más cercana al minuto :00 si hay varias).
    """
    file = f"{folder_path}/{iata}_2023.csv"
    df_raw = pd.read_csv(file, low_memory=False)

    df_raw['DATE'] = pd.to_datetime(df_raw['DATE'])
    df_raw['fecha'] = df_raw['DATE'].dt.date
    df_raw['hora']  = df_raw['DATE'].dt.hour

    df_raw['temperatura_c']    = df_raw['TMP'].apply(parse_escalated_value)
    df_raw['punto_rocio_c']    = df_raw['DEW'].apply(parse_escalated_value)
    df_raw['presion_hpa']      = df_raw['SLP'].apply(parse_escalated_value)
    df_raw['viento_ms']        = df_raw['WND'].apply(parse_wind)
    df_raw['visibilidad_m']    = df_raw['VIS'].apply(lambda x: parse_escalated_value(x, scale=1))
    df_raw['precipitacion_mm'] = df_raw['AA1'].apply(parse_precipitation) if 'AA1' in df_raw.columns else 0.0

    # Nos quedamos con un registro por hora (el primero disponible)
    df_date = (df_raw
                  .sort_values('DATE')
                  .drop_duplicates(subset=['fecha', 'hora'], keep='first'))

    df_date['IATA'] = iata

    finale_columns = ['IATA', 'fecha', 'hora', 'temperatura_c', 'punto_rocio_c',
                         'presion_hpa', 'viento_ms', 'visibilidad_m', 'precipitacion_mm']

    return df_date[finale_columns]


# Procesamos los 49 aeropuertos y concatenamos
dfs_climate = []
for iata in noaa_stations.keys():
    try:
        df_iata = clean_noaa(iata)
        dfs_climate.append(df_iata)
        print(f"{iata}: {len(df_iata)} registros horarios procesados.")
    except Exception as e:
        print(f"{iata} error al procesar: {e}")

df_climate = pd.concat(dfs_climate, ignore_index=True)

print(f"\nTotal de registros horarios de clima: {len(df_climate):,}")
print(f"Missings por columna:\n{df_climate.isna().sum()}")
df_climate.head()

# 2.3 Reconstrucción de la hora programada y cruce con NOAA

In [ ]:
def hhmm_to_min(hhmm):
    """Convierte formato HHMM (ej. 830, 2355) a minutos desde medianoche."""
    if pd.isna(hhmm):
        return np.nan
    hhmm = int(hhmm)
    if hhmm == 2400:
        hhmm = 0
    hour = hhmm // 100
    minutes = hhmm % 100
    return hour * 60 + minutes

# Separamos vuelos cancelados (no tienen DEP_TIME, no se puede reconstruir su hora programada)
df_canceled = df[df['CANCELLED'] == 1].copy()
df_operated   = df[df['CANCELLED'] == 0].copy()

print(f"Vuelos operados: {len(df_operated):,}")
print(f"Vuelos cancelados: {len(df_canceled):,}")

df_operated['dep_minutos'] = df_operated['DEP_TIME'].apply(hhmm_to_min)
df_operated['crs_minutos'] = (df_operated['dep_minutos'] - df_operated['DEP_DELAY']) % 1440
df_operated['hora_programada'] = (df_operated['crs_minutos'] // 60).astype(int)

df_operated = df_operated[df_operated['ORIGIN'].isin(top_50)].copy()
df_operated['fecha'] = pd.to_datetime(df_operated['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p').dt.date
df_operated = df_operated.rename(columns={'ORIGIN': 'IATA', 'hora_programada': 'hora'})

print(f"\nVuelos tras filtrar por los 49 aeropuertos: {len(df_operated):,}")

# CRUCE FINAL: vuelos (BTS) + clima (NOAA)

df_final = df_operated.merge(
    df_climate,
    on=['IATA', 'fecha', 'hora'],
    how='left'
)

print(f"\nRegistros tras el cruce: {len(df_final):,}")
print(f"Vuelos sin dato meteorológico emparejado: {df_final['temperatura_c'].isna().sum():,} "
      f"({df_final['temperatura_c'].isna().mean():.1%})")

df_final.head()

# 2.4 Definición de la variable objetivo

In [ ]:
df_final['retraso'] = (df_final['DEP_DELAY'] > 15).astype(int)

print(df_final['retraso'].value_counts())
print(df_final['retraso'].value_counts(normalize=True).round(3))

# 3.1 Estadísticas descriptivas

In [ ]:
print("Dimensiones del dataset final:", df_final.shape)
print("\nTipos de datos:")
print(df_final.dtypes)

print("\nEstadísticos descriptivos (variables numéricas):")
df_final[['DEP_DELAY', 'ARR_DELAY', 'DISTANCE', 'temperatura_c',
           'viento_ms', 'visibilidad_m', 'precipitacion_mm']].describe()

# 3.2 Distribución de retrasos por aerolínea, aeropuerto, mes y franja horaria

In [ ]:
df_final['mes'] = pd.to_datetime(df_final['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p').dt.month


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Por aerolínea ---
tasa_aerolinea = df_final.groupby('OP_UNIQUE_CARRIER')['retraso'].mean().sort_values(ascending=False)
tasa_aerolinea.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Tasa de retraso por aerolínea')
axes[0, 0].set_ylabel('% de vuelos retrasados')
axes[0, 0].tick_params(axis='x', rotation=90)

# --- Por aeropuerto ---
tasa_aeropuerto = df_final.groupby('IATA')['retraso'].mean().sort_values(ascending=False)
tasa_aeropuerto.head(15).plot(kind='bar', ax=axes[0, 1], color='indianred')
axes[0, 1].set_title('Top 15 aeropuertos con mayor tasa de retraso')
axes[0, 1].set_ylabel('% de vuelos retrasados')
axes[0, 1].tick_params(axis='x', rotation=90)

# --- Por mes ---
tasa_mes = df_final.groupby('mes')['retraso'].mean()
tasa_mes.plot(kind='line', marker='o', ax=axes[1, 0], color='darkgreen')
axes[1, 0].set_title('Tasa de retraso por mes')
axes[1, 0].set_xlabel('Mes')
axes[1, 0].set_ylabel('% de vuelos retrasados')
axes[1, 0].set_xticks(range(1, 13))

# --- Por franja horaria ---
tasa_hora = df_final.groupby('hora')['retraso'].mean()
tasa_hora.plot(kind='bar', ax=axes[1, 1], color='goldenrod')
axes[1, 1].set_title('Tasa de retraso por hora programada de salida')
axes[1, 1].set_xlabel('Hora del día')
axes[1, 1].set_ylabel('% de vuelos retrasados')

plt.tight_layout()
plt.show()

print("\nAerolínea con mayor tasa de retraso:", tasa_aerolinea.index[0], f"({tasa_aerolinea.iloc[0]:.1%})")
print("Aeropuerto con mayor tasa de retraso:", tasa_aeropuerto.index[0], f"({tasa_aeropuerto.iloc[0]:.1%})")
print("Mes con mayor tasa de retraso:", tasa_mes.idxmax(), f"({tasa_mes.max():.1%})")
print("Hora con mayor tasa de retraso:", tasa_hora.idxmax(), f"({tasa_hora.max():.1%})")

In [ ]:
# Verificamos si la hora 2am es un patrón real o ruido
hour_count = df_final.groupby('hora').size()
print(f"\nVuelos programados a las 2am: {hour_count.get(2, 0):,} "
      f"({hour_count.get(2, 0) / len(df_final):.2%} del total)")
# Verificamos si hay mas horas con menos de mil vuelos
print("Horas con menos de 1,000 vuelos programados (no representativas):")
print(hour_count[hour_count < 1000])

In [ ]:
# Creamos franjas horariasm para resolver el ruido generado por horas con pocos vuelos
def asignar_franja(hora):
    if 0 <= hora <= 5:
        return 'Madrugada (00-05h)'
    elif 6 <= hora <= 11:
        return 'Mañana (06-11h)'
    elif 12 <= hora <= 17:
        return 'Tarde (12-17h)'
    else:
        return 'Noche (18-23h)'

df_final['franja_horaria'] = df_final['hora'].apply(asignar_franja)

print(df_final['franja_horaria'].value_counts())

## 3.2.1. Tasa de retraso por franja horaria

In [ ]:
band_order = ['Madrugada (00-05h)', 'Mañana (06-11h)', 'Tarde (12-17h)', 'Noche (18-23h)']

band_rate = df_final.groupby('franja_horaria')['retraso'].mean().reindex(band_order)

fig, ax = plt.subplots(figsize=(7, 5))
band_rate.plot(kind='bar', ax=ax, color='goldenrod')
ax.set_title('Tasa de retraso por franja horaria de salida')
ax.set_ylabel('% de vuelos retrasados')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print(band_rate.round(3))

# 3.3 Análisis de causas de retraso declaradas

In [ ]:
columnas_causa = ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
                   'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

df_retrasados = df_final[df_final['retraso'] == 1].copy()

print(f"Vuelos con retraso >15 min: {len(df_retrasados):,}\n")

causes_sum = df_retrasados[columnas_causa].sum().sort_values(ascending=False)
print("Minutos totales de retraso por causa:")
print(causes_sum)

df_retrasados['causa_principal'] = df_retrasados[columnas_causa].idxmax(axis=1)
print("\nCausa principal - Porcentaje de vuelos retrasados:")
print(df_retrasados['causa_principal'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(8, 5))
df_retrasados['causa_principal'].value_counts(normalize=True).plot(kind='bar', ax=ax, color='teal')
ax.set_title('Causa principal de retraso (BTS)')
ax.set_ylabel('Porcentaje de vuelos retrasados')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 3.4 Relación entre variables meteorológicas y retrasos

## 3.4.1. Matriz de correlación entre clima y DEP_DELAY

In [ ]:
variables_climate = ['temperatura_c', 'punto_rocio_c', 'presion_hpa',
                    'viento_ms', 'visibilidad_m', 'precipitacion_mm']

corr_matrix = df_final[variables_climate + ['DEP_DELAY']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlación entre variables meteorológicas y retraso de salida')
plt.tight_layout()
plt.show()

print("Correlación de cada variable con DEP_DELAY:")
print(corr_matrix['DEP_DELAY'].drop('DEP_DELAY').sort_values(key=abs, ascending=False))

## 3.4.2. Tasa de retraso por rangos de visibilidad y precipitación

In [ ]:
df_final['bin_visibilidad'] = pd.cut(
    df_final['visibilidad_m'],
    bins=[-1, 1000, 5000, 10000, 200000],
    labels=['Muy baja (<1km)', 'Baja (1-5km)', 'Media (5-10km)', 'Buena (>10km)']
)

df_final['bin_precipitacion'] = pd.cut(
    df_final['precipitacion_mm'],
    bins=[-0.1, 0, 2, 10, 200],
    labels=['Sin lluvia', 'Leve (0-2mm)', 'Moderada (2-10mm)', 'Fuerte (>10mm)']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_final.groupby('bin_visibilidad', observed=True)['retraso'].mean().plot(
    kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Tasa de retraso por visibilidad')
axes[0].set_ylabel('% de vuelos retrasados')
axes[0].tick_params(axis='x', rotation=20)

df_final.groupby('bin_precipitacion', observed=True)['retraso'].mean().plot(
    kind='bar', ax=axes[1], color='cadetblue')
axes[1].set_title('Tasa de retraso por intensidad de precipitación')
axes[1].set_ylabel('% de vuelos retrasados')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print(df_final.groupby('bin_visibilidad', observed=True)['retraso'].agg(['mean', 'count']))
print()
print(df_final.groupby('bin_precipitacion', observed=True)['retraso'].agg(['mean', 'count']))

## 3.4.3. Viento fuerte. Umbral >12.9 m/s

In [ ]:
df_final['viento_fuerte'] = (df_final['viento_ms'] > 12.9).astype(int)

wind_rate = df_final.groupby('viento_fuerte')['retraso'].agg(['mean', 'count'])
wind_rate.index = ['Viento normal', 'Viento fuerte (>12.9 m/s)']
print(wind_rate)

# 3.5 Nulos, outliers y anomalías

In [ ]:
summary_nulls = df_final.isna().sum()
summary_nulls = summary_nulls[summary_nulls > 0].sort_values(ascending=False)
print("Resumen de valores nulos (columnas con al menos 1 NaN):")
print(summary_nulls)
print(f"\n% sobre el total de filas ({len(df_final):,}):")
print((summary_nulls / len(df_final) * 100).round(2))

# Outliers de DEP_DELAY (detectados en 3.1)
p99 = df_final['DEP_DELAY'].quantile(0.99)
print(f"\nPercentil 99 de DEP_DELAY: {p99:.0f} minutos")
print(f"Vuelos por encima del P99: {(df_final['DEP_DELAY'] > p99).sum():,} "
      f"({(df_final['DEP_DELAY'] > p99).mean():.2%})")

In [ ]:
df_outliers = df_final[df_final['DEP_DELAY'] > 201].copy()

print(f"Vuelos con retraso extremo (>201 min): {len(df_outliers):,} ({len(df_outliers)/len(df_final):.2%})\n")

# --- Aerolíneas ---
print("Top 10 aerolíneas en outliers - peso normal:")
print(df_outliers['OP_UNIQUE_CARRIER'].value_counts(normalize=True).head(10).round(3))

print("\nTop 10 aeropuertos en outliers:")
print(df_outliers['IATA'].value_counts(normalize=True).head(10).round(3))

# --- Meses ---
df_outliers['mes'] = pd.to_datetime(df_outliers['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p').dt.month
print("\nDistribución por mes de los outliers:")
print(df_outliers['mes'].value_counts(normalize=True).sort_index().round(3))

# --- Condiciones climáticas severas ---
print("\nComparación de condiciones climáticas: outliers vs. resto del dataset")
comparason = pd.DataFrame({
    'outliers (>201 min)': df_outliers[['viento_ms', 'precipitacion_mm', 'visibilidad_m']].mean(),
    'resto del dataset': df_final[df_final['DEP_DELAY'] <= 201][['viento_ms', 'precipitacion_mm', 'visibilidad_m']].mean()
})
print(comparason.round(2))

# --- Causa principal ---
causes_outliers = df_outliers[['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
                                 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']].idxmax(axis=1)
print("\nCausa principal declarada en los vuelos con retraso extremo:")
print(causes_outliers.value_counts(normalize=True).round(3))

# 4.1 Tratamiento de nulos y variables irrelevantes

In [ ]:
columns_to_exclude = [
    'CANCELLATION_CODE',                                            # 100% nula, vuelos no cancelados
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',                        # fuga de datos (post-hoc)
    'presion_hpa',                                                  # descartada: 10.28% nulos, corr. débil (-0.052)
    'DEP_TIME', 'DEP_DELAY', 'ARR_TIME', 'ARR_DELAY',               # fuga de datos (posteriores al hecho)
    'dep_minutos', 'crs_minutos',                                   # variables auxiliares intermedias, ya no se necesitan
    'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID',
    'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID'  # IDs redundantes con IATA/DEST
]

df_model = df_final.drop(columns=columns_to_exclude)

rows_before = len(df_model)
df_model = df_model.dropna(subset=['viento_ms', 'punto_rocio_c', 'visibilidad_m',
                                       'temperatura_c', 'precipitacion_mm'])
rows_after = len(df_model)

print(f"Filas antes de limpiar nulos residuales: {rows_before:,}")
print(f"Filas después: {rows_after:,}")
print(f"Filas eliminadas: {rows_before - rows_after:,} ({(rows_before-rows_after)/rows_before:.2%})")

print(f"\nColumnas finales del dataset de modelado ({df_model.shape[1]}):")
print(df_model.columns.tolist())

print(f"\nNulos restantes:")
print(df_model.isna().sum().sum())

In [ ]:
df_model = df_model.drop(columns=['CANCELLED'])
print(df_model.columns.tolist())

# 4.2 Creación de variables derivadas

## 4.2.1. Indicador de temporada alta

In [ ]:
reference_month_rate = df_final.groupby('mes')['retraso'].mean()
general_mean = reference_month_rate.mean()
peak_season_months = reference_month_rate[reference_month_rate > general_mean].index.tolist()

df_model['temporada_alta'] = df_model['mes'].isin(peak_season_months).astype(int)

print(f"Tasa de retraso promedio general: {general_mean:.3f}")
print(f"Meses por encima de la media (temporada alta): {sorted(peak_season_months)}")
print("\nTemporada alta - distribución:")
print(df_model['temporada_alta'].value_counts())
print("\nTasa de retraso: temporada alta vs. resto")
print(df_model.groupby('temporada_alta')['retraso'].mean().round(3))

## 4.2.2. Índice de congestión del aeropuerto por hora

In [ ]:
if 'indice_congestion' in df_model.columns:
    df_model = df_model.drop(columns=['indice_congestion'])

congestion = (df_model
              .groupby(['IATA', 'fecha', 'hora'])
              .size()
              .reset_index(name='indice_congestion'))
df_model = df_model.merge(congestion, on=['IATA', 'fecha', 'hora'], how='left')

print("Estadísticos del índice de congestión:")
print(df_model['indice_congestion'].describe())
print(f"\nCorrelación con retraso: {df_model['indice_congestion'].corr(df_model['retraso']):.3f}")

# 4.3 Codificación de variables categóricas

In [ ]:
# Variables ordinales
time_slot_order = [['Madrugada (00-05h)', 'Mañana (06-11h)', 'Tarde (12-17h)', 'Noche (18-23h)']]
visibility_order = [['Muy baja (<1km)', 'Baja (1-5km)', 'Media (5-10km)', 'Buena (>10km)']]
precipitation_order = [['Sin lluvia', 'Leve (0-2mm)', 'Moderada (2-10mm)', 'Fuerte (>10mm)']]

df_model['franja_horaria_enc'] = OrdinalEncoder(categories=time_slot_order).fit_transform(df_model[['franja_horaria']])
df_model['bin_visibilidad_enc'] = OrdinalEncoder(categories=visibility_order).fit_transform(df_model[['bin_visibilidad']])
df_model['bin_precipitacion_enc'] = OrdinalEncoder(categories=precipitation_order).fit_transform(df_model[['bin_precipitacion']])

# Variables de alta cardinalidad
for col in ['OP_UNIQUE_CARRIER', 'IATA', 'DEST']:
    freq = df_model[col].value_counts(normalize=True)
    df_model[f'{col}_freq'] = df_model[col].map(freq)

print(df_model[['franja_horaria', 'franja_horaria_enc',
                  'bin_visibilidad', 'bin_visibilidad_enc',
                  'OP_UNIQUE_CARRIER', 'OP_UNIQUE_CARRIER_freq']].head(10))

# 4.4 Análisis de desbalance de clases y estrategia de tratamiento

In [ ]:
print("Distribución final de la variable objetivo:")
print(df_model['retraso'].value_counts())
print(df_model['retraso'].value_counts(normalize=True).round(3))

ratio = df_model['retraso'].value_counts()[0] / df_model['retraso'].value_counts()[1]
print(f"\nRatio de desbalance: {ratio:.1f}:1")

In [ ]:
# Se eliminan duplicados de indice_congestion
columns_congestion_duplicates = [c for c in df_model.columns if c.startswith('indice_congestion')]
print(f"Columnas de congestión encontradas: {columns_congestion_duplicates}")

if len(columns_congestion_duplicates) > 1:
    df_model = df_model.drop(columns=[c for c in columns_congestion_duplicates if c != 'indice_congestion_x'])
    df_model = df_model.rename(columns={'indice_congestion_x': 'indice_congestion'})

print(f"Columnas de congestión tras limpiar: {[c for c in df_model.columns if 'congestion' in c]}")

# 5.1 Separación train/test y estrategia de validación

In [ ]:
columns_to_exclude_end = [
    'FL_DATE', 'fecha', 'mes',                                   # ya capturadas en variables derivadas
    'franja_horaria', 'bin_visibilidad', 'bin_precipitacion',    # versiones texto, ya tenemos las _enc
    'OP_UNIQUE_CARRIER_freq', 'IATA_freq', 'DEST_freq',          # se reemplazan por target encoding
    'retraso'                                                    # es el target
]
columns_to_exclude_end = [c for c in columns_to_exclude_end if c in df_model.columns]

X = df_model.drop(columns=columns_to_exclude_end)
y = df_model['retraso']

print(f"Features finales antes de target encoding ({X.shape[1]}):")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\nTrain: {X_train.shape[0]:,} filas")
print(f"Test:  {X_test.shape[0]:,} filas")
print(f"Proporción de retraso en train: {y_train.mean():.3f}")
print(f"Proporción de retraso en test:  {y_test.mean():.3f}")

In [ ]:
def apply_target_encoding(X_train, X_test, column, target_train, suavizado=20):
    """
    Target encoding con suaviazado bayesiano:
    Evita sobreajuste en categorías con pocos casos.
    """
    global_mean = target_train.mean()
    stats = target_train.groupby(X_train[column]).agg(['mean', 'count'])
    stats.columns = ['media_categoria', 'conteo']
    stats['encoded'] = (
        (stats['media_categoria'] * stats['conteo'] + global_mean * suavizado)
        / (stats['conteo'] + suavizado)
    )
    map = stats['encoded'].to_dict()
    train_encoded = X_train[column].map(map)
    test_encoded = X_test[column].map(map).fillna(global_mean)
    return train_encoded, test_encoded

for col in ['OP_UNIQUE_CARRIER', 'IATA', 'DEST']:
    X_train[f'{col}_target_enc'], X_test[f'{col}_target_enc'] = apply_target_encoding(
        X_train, X_test, col, y_train
    )

# Quitamos las columnas de texto originales, dado que ya han sido codificadas
X_train = X_train.drop(columns=['OP_UNIQUE_CARRIER', 'IATA', 'DEST'])
X_test = X_test.drop(columns=['OP_UNIQUE_CARRIER', 'IATA', 'DEST'])

print("Columnas finales del set de entrenamiento:")
print(X_train.columns.tolist())
print(f"\nEjemplo de target encoding (aerolínea):")
print(X_train['OP_UNIQUE_CARRIER_target_enc'].describe())

# 5.2 Modelo 1: Regresión Logística

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

start_time = time.time()

model_lr = LogisticRegression(
    class_weight='balanced',   # compensa el desbalance 79/21 (sección 4.5)
    max_iter=1000,
    random_state=42
)
model_lr.fit(X_train_scaled, y_train)

print(f"Tiempo de entrenamiento: {time.time() - start_time:.1f} segundos")

y_pred_lr = model_lr.predict(X_test_scaled)
y_proba_lr = model_lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nAUC-ROC: {roc_auc_score(y_test, y_proba_lr):.4f}")
print(f"F1-score (clase 'retraso'): {f1_score(y_test, y_pred_lr):.4f}")
print("\n", classification_report(y_test, y_pred_lr, target_names=['Sin retraso', 'Con retraso']))

coeficiente = pd.Series(model_lr.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print("\ncoeficiente ordenados por magnitud (efecto sobre la probabilidad de retraso):")
print(coeficiente)

# 5.3 Random Forest

In [ ]:
params_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15],
    'min_samples_leaf': [500, 1000] # Necesario que sea alto para un dataset de 4.2M
}

start_time = time.time()

rf_base = RandomForestClassifier(
    class_weight='balanced',
    max_samples=0.2, # Es necesario que sea alto para un dataset de 4.2M
    n_jobs=1,
    random_state=42
)

grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=params_rf,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

print(f"Tiempo total de búsqueda: {(time.time() - start_time) / 60:.1f} minutos")
print(f"\nMejores parámetros encontrados: {grid_search_rf.best_params_}")

model_rf = grid_search_rf.best_estimator_

In [ ]:
y_train_pred_rf = model_rf.predict(X_train)
y_test_pred_rf = model_rf.predict(X_test)

print(f"\nAccuracy en train: {accuracy_score(y_train, y_train_pred_rf):.4f}")
print(f"Accuracy en test:  {accuracy_score(y_test, y_test_pred_rf):.4f}")

y_proba_rf = model_rf.predict_proba(X_test)[:, 1]
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"F1-score (clase 'retraso'): {f1_score(y_test, y_test_pred_rf):.4f}")
print("\n", classification_report(y_test, y_test_pred_rf, target_names=['Sin retraso', 'Con retraso']))

# 5.4 XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

params_xgb = {
    'n_estimators': [100, 200],
    'eta': [0.1, 0.3],
    'gamma': [0, 1],
    'max_depth': [5, 6],
}

start_time = time.time()

xgb_base = xgb.XGBClassifier(
    booster='gbtree',
    tree_method='hist',
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    n_jobs=-1,
    random_state=42
)

grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=params_xgb,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search_xgb.fit(X_train, y_train)

print(f"Tiempo total de búsqueda: {(time.time()-start_time)/60:.1f} minutos")
print(f"\nMejores parámetros encontrados: {grid_search_xgb.best_estimator_}")

model_xgb = grid_search_xgb.best_estimator_

y_train_pred_xgb = model_xgb.predict(X_train)
y_test_pred_xgb = model_xgb.predict(X_test)
print(f"\nAccuracy en train: {accuracy_score(y_train, y_train_pred_xgb):.4f}")
print(f"Accuracy en test:  {accuracy_score(y_test, y_test_pred_xgb):.4f}")

y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_proba_xgb):.4f}")
print(f"F1-score (clase 'retraso'): {f1_score(y_test, y_test_pred_xgb):.4f}")
print("\n", classification_report(y_test, y_test_pred_xgb, target_names=['Sin retraso', 'Con retraso']))

# 6.1 Curva ROC y matriz de confusión — modelo XGBoost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

fpr, tpr, thresholds = roc_curve(y_test, y_proba_xgb)
axes[0].plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc_score(y_test, y_proba_xgb):.3f})', color='darkorange')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Modelo aleatorio')
axes[0].set_xlabel('Tasa de falsos positivos')
axes[0].set_ylabel('Tasa de verdaderos positivos (Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend()

cm = confusion_matrix(y_test, y_test_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sin retraso', 'Con retraso'])
disp.plot(ax=axes[1], cmap='Blues', values_format=',')
axes[1].set_title('Matriz de confusión (umbral = 0.5)')

plt.tight_layout()
plt.show()

print(f"Verdaderos negativos: {cm[0,0]:,}")
print(f"Falsos positivos:     {cm[0,1]:,}")
print(f"Falsos negativos:     {cm[1,0]:,}")
print(f"Verdaderos positivos: {cm[1,1]:,}")

# 6.2 Análisis de umbral de decisión óptimo

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
resultados_umbral = []

for u in thresholds:
    y_pred_u = (y_proba_xgb >= u).astype(int)
    resultados_umbral.append({
        'umbral': u,
        'precision': precision_score(y_test, y_pred_u),
        'recall': recall_score(y_test, y_pred_u),
        'f1': f1_score(y_test, y_pred_u)
    })

df_thresholds = pd.DataFrame(resultados_umbral)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(df_thresholds['umbral'], df_thresholds['precision'], marker='o', label='Precision')
ax.plot(df_thresholds['umbral'], df_thresholds['recall'], marker='o', label='Recall')
ax.plot(df_thresholds['umbral'], df_thresholds['f1'], marker='o', label='F1-score', linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='Umbral por defecto (0.5)')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Score')
ax.set_title('Trade-off Precision/Recall según el umbral de decisión')
ax.legend()
plt.tight_layout()
plt.show()

print(df_thresholds.round(3))

umbral_optimo_f1 = df_thresholds.loc[df_thresholds['f1'].idxmax(), 'umbral']
print(f"\nUmbral que maximiza F1: {umbral_optimo_f1:.2f}")

# Umbral con Recall >= 0.75 para no perdernos retrasos reales
candidatos_recall_alto = df_thresholds[df_thresholds['recall'] >= 0.75]
if len(candidatos_recall_alto) > 0:
    umbral_recall_alto = candidatos_recall_alto.loc[candidatos_recall_alto['precision'].idxmax(), 'umbral']
    print(f"Umbral con Recall≥0.75 que maximiza Precision: {umbral_recall_alto:.2f}")

In [ ]:
THRESHOLD_CHOISE = 0.40

y_pred_final = (y_proba_xgb >= THRESHOLD_CHOISE).astype(int)

print(f"Métricas finales del modelo XGBoost con umbral = {THRESHOLD_CHOISE}:")
print(classification_report(y_test, y_pred_final, target_names=['Sin retraso', 'Con retraso']))

cm_final = confusion_matrix(y_test, y_pred_final)
print(f"\nMatriz de confusión (umbral={THRESHOLD_CHOISE}):")
print(f"Verdaderos negativos: {cm_final[0,0]:,}")
print(f"Falsos positivos:     {cm_final[0,1]:,}")
print(f"Falsos negativos:     {cm_final[1,0]:,}")
print(f"Verdaderos positivos: {cm_final[1,1]:,}")

# 7.1 Cálculo de valores SHAP

---



In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=100_000, replace=False)
X_test_sample = X_test.loc[sample_idx]

dmatrix_sample = xgb.DMatrix(X_test_sample)

shap_values = model_xgb.get_booster().predict(dmatrix_sample, pred_contribs=True) # pred_contribs=True devuelve la contribución SHAP de cada variable a cada predicción
shap_values = shap_values[:, :-1]  # quitamos la columna de bias para quedarnos solo con las variables

print(f"Shape de shap_values: {shap_values.shape}")
print("Cálculo de SHAP completado (vía XGBoost nativo).")

# 7.2 Importancia de variables sengún SHAP

In [ ]:
importance_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test_sample.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importance_shap.plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Impacto medio en la predicción (|SHAP value|)')
ax.set_title('Importancia de variables según SHAP')
plt.tight_layout()
plt.show()

print(importance_shap)

# 7.3 Peso conjunto de las variables climáticas - SHAP

In [ ]:
variables_climatic = ['temperatura_c', 'punto_rocio_c', 'viento_ms',
                          'visibilidad_m', 'precipitacion_mm']

climatic_weight = importance_shap[variables_climatic].sum()
total_weight = importance_shap.sum()

print(f"Peso conjunto de variables climáticas (SHAP): {climatic_weight:.3f}")
print(f"Peso total de todas las variables: {total_weight:.3f}")
print(f"Proporción del peso climático sobre el total: {climatic_weight/total_weight:.1%}")

# 7.4 Aeropuertos y aerolíneas más sensibles al mal tiempo

In [ ]:
climatic_columns = ['temperatura_c', 'punto_rocio_c', 'viento_ms', 'visibilidad_m', 'precipitacion_mm']
climatic_indexes = [X_test_sample.columns.get_loc(c) for c in climatic_columns]

climatic_sensitivity = shap_values[:, climatic_indexes].sum(axis=1)

# Recuperamos el aeropuerto y la aerolínea reales de cada vuelo de la muestra
# (df_modelo todavía conserva las columnas de texto originales, X_test ya no)
info_sample = df_model.loc[X_test_sample.index, ['IATA', 'OP_UNIQUE_CARRIER']].copy()
info_sample['sensibilidad_climatica'] = climatic_sensitivity

# --- Por aeropuerto ---
airport_sensitivity = (info_sample.groupby('IATA')['sensibilidad_climatica']
                            .mean().sort_values(ascending=False))

# --- Por aerolínea ---
airline_sensitivity = (info_sample.groupby('OP_UNIQUE_CARRIER')['sensibilidad_climatica']
                           .mean().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

airport_sensitivity.head(10).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title('Top 10 aeropuertos más sensibles al clima (SHAP)')
axes[0].set_xlabel('Contribución media del clima a la predicción')

airline_sensitivity.plot(kind='barh', ax=axes[1], color='indianred')
axes[1].invert_yaxis()
axes[1].set_title('Sensibilidad al clima por aerolínea (SHAP)')
axes[1].set_xlabel('Contribución media del clima a la predicción')

plt.tight_layout()
plt.show()

print("Top 10 aeropuertos más sensibles al clima:")
print(airport_sensitivity.head(10))
print("\nTop 10 aeropuertos menos sensibles al clima:")
print(airport_sensitivity.tail(10))
print("\nSensibilidad por aerolínea (todas):")
print(airline_sensitivity)

# 8.1 Serialización del modelo entrenado

In [ ]:
pickle.dump(model_xgb, open('modelo_retrasos.pkl', 'wb'))

def calculate_map_encoding(X_train, column, target_train, suavizado=20):
    global_average = target_train.mean()
    stats = target_train.groupby(X_train[column]).agg(['mean', 'count'])
    stats.columns = ['media_categoria', 'conteo']
    stats['encoded'] = (
        (stats['media_categoria'] * stats['conteo'] + global_average * suavizado)
        / (stats['conteo'] + suavizado)
    )
    return stats['encoded'].to_dict(), global_average

map_carrier, global_average_carrier = calculate_map_encoding(
    df_model.loc[X_train.index], 'OP_UNIQUE_CARRIER', y_train)
map_iata, global_average_iata = calculate_map_encoding(
    df_model.loc[X_train.index], 'IATA', y_train)
map_dest, global_average_dest = calculate_map_encoding(
    df_model.loc[X_train.index], 'DEST', y_train)

encoders = {
    'carrier': {'mapa': map_carrier, 'media_global': global_average_carrier},
    'iata': {'mapa': map_iata, 'media_global': global_average_iata},
    'dest': {'mapa': map_dest, 'media_global': global_average_dest},
    'columns_modelo': X_train.columns.tolist(),
    'umbral_decision': 0.40
}

pickle.dump(encoders, open('encoders_retrasos.pkl', 'wb'))